# INFEWS Farm Model

## Objective and Prerequisites

This model is based of the example of a stochastic optimization problem from [Gurobi](https://www.gurobi.com/resource/solving-simple-stochastic-optimization-problems-with-gurobi/). That problem asked you to consider a situation where you have to buy today the stock of products to sell tomorrow but you don't know the exact demand you will see. Then you will have to decide what to do with whatever is left which includes selling it for scrap. This is called the Newspaper Vendor Problem. 


---
## Problem Description

For our problem a farm must decide how to prepare for different climate futures. Each climate future has different precipation patterns that will affect how much irrigation water is needed for different plants. Our farm must decide if and how much alternative water and electricity to invest in. Using a combination of precipation, alternative water, and irrigation water the decision makers maximizes profit where revenue is based on the price of the crop and the yield available and the costs are the costs of investing in alternative water and buying irrigation water. There are known costs associated with buying irrigation water, investing in alternative water, but the unknown is knowing how much water will be available as rain.


---
## Python Implementation

We import the Gurobi Python Module and other Python libraries.

Import Libraries, Helpful Links, and Making Sure all Results show after each chunk.

In [ ]:
from gurobipy import * 
import gurobipy as gp
import random
random.seed(a=100)
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import gridspec
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import scipy.stats as scs

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import time



# Problem Source: https://www.gurobi.com/resource/solving-simple-stochastic-optimization-problems-with-gurobi/
# Matplotlib documentation: https://matplotlib.org/tutorials/introductory/pyplot.html
# Pandas documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html

In [ ]:
# CSV improts

rain_c0_df = pd.read_csv('precips_c0_dml.csv', index_col=0)

rain_c0_dict = rain_c0_df.set_index(['run','year']).T.to_dict('records')

rain_c0_dict = rain_c0_dict[0]





In [ ]:
# EP EV rain
rain_dict1 = {1: 8.89, 2: 8.89, 3: 8.89, 
              4: 26.67, 5: 26.67, 6: 26.67, 7: 26.67, 8: 26.67, 9: 26.67, 
              10: 53.34, 11: 53.34, 12: 53.34, 13: 53.34, 14: 53.34, 15: 53.34, 16: 53.34, 17: 53.34,
             18: 80.01, 19: 80.01, 20: 80.01, 21: 80.01, 22: 80.01,
             23: 107, 24: 107, 25: 107}

# DML EV rain
rain_dict2 = {1: 8.89, 2: 8.89, 3: 8.89, 4: 8.89, 
              5: 26.67, 6: 26.67, 7: 26.67, 8: 26.67, 9: 26.67, 10: 26.67, 11: 26.67, 12: 26.67, 13: 26.67, 14: 26.67, 
              15: 53.34, 16: 53.34, 17: 53.34, 18: 53.34, 19: 53.34, 20: 53.34, 21: 53.34, 22: 53.34,
             23: 80.01, 24: 80.01, 
             25: 107}

Some helpful conversions.
And some parameterization and upper and lower bounds.

In [ ]:
# Conversions

cm2_ha = 100000000 # 100,000,000 cm2 = 1 ha

cm3_gal = 3785.41 # 3785.41 cm2 = 1 Gal

m3_kGal = 3.785411784

gal_acre_ft = 325851 # 1 acre ft = 325851 gallons

acre_ha = 2.47105 # 1 hectare = 2.47105 acres

cm_ft = 30.48 # 1 ft = 30.48 cm

cm_in = 2.54

lb_tonne = 2204.62

# Hours in a Month
hrs_month = 720

#Days in Month
days_month = 30

#Growing Season Months
gs_months = 9

# Parameters

ha = 200 # hectares average size of texas farms 411 acres rounded up, https://www.texasagriculture.gov/About/TexasAgStats.aspx#:~:text=For%2036%25%20of%20producers%20in,by%2012%20acres%20from%202012.

price = 200 # $/ metric ton #https://www.indexmundi.com/commodities/?commodity=wheat

#note $ /acre ranges from $7-85 / acre assuming only a few cm of depth similiar price to irrigation water 


#Water production cost $2 / Gal convert to $ / cm
cost_alt_water = 2 / 365 / cm3_gal * cm2_ha * ha


# 6.25 kWh / kGal > / Gal > / acre-ft > acre-cm > / ha-cm > kwh/cm
alt_water_elc = 6.25 / 1000 * gal_acre_ft / cm_ft * acre_ha * ha







# Irrigation Water Cost https://news.berkeley.edu/berkeley_blog/the-cost-of-irrigation-water-and-urban-farming/
#  Given in $ / cm from: $ 40 / acre-ft / gal_acre_ft / cm3_gal * cm2_ha * ha 

#cost_irrigation_water = 40 / gal_acre_ft / cm3_gal * cm2_ha * ha  # $ / cm 


# $7 / ac- in convert to $ / cm
cost_irrigation_water =  7 / cm_in * acre_ha * ha

# $10 / ac- in convert to $ / cm
cost_op_alt_water =  10 / cm_in * acre_ha * ha


# 1 kWh / kGal
irrigation_water_elc = 1 / 1000 * gal_acre_ft / cm_ft * acre_ha * ha


salt_conc = 2 # deciSiemens / meter = dS/m

salt_conc_water = 1 # deciSiemens / meter = dS/m

precip = 20 * cm_in # 20 inches of rain


 #  1.5 acre-ft / acre / yr halved for season https://www.nass.usda.gov/Publications/Highlights/2019/2017Census_Irrigation_and_WaterManagement.pdf
 

irrigation_water_limit = 0.25

max_irrigation_water = irrigation_water_limit * cm_ft 

# $ / kW of capacity
cost_alt_elc = 1500



cost_utility_elc = 0.08



#Renewable Energy

# Capacity Factor Solar https://pv-magazine-usa.com/2018/07/03/texas-getting-315-mw-of-solar-power/

cf_solar = 0.30

# Fu, R., Feldman, D., Margolis, R., 2018. U.S. Solar Photovoltaic System Cost Benchmark: Q1 2018. Tech. rep., National Renewable Energy Laboratory, Golden, CO.

solar_dollar_kW = 1500

# wInd Costs https://www.energy.gov/sites/prod/files/2019/08/f65/2018%20Wind%20Technologies%20Market%20Report%20FINAL.pdf

wind_dollar_kW = 840

Seeting the precipatation values.

This is where I can create climate regimes. 

Have certain climate regimes have certain values of precipatation

A quick chart showing the effect of water depth on yield and price.

In [ ]:
water = np.array(range(120))


# From Dinar et. al 1991 "Production functions relating crop yield, water quality and quantity, soil salinity and drainage volume"

y1 = (-3.350 + 0.2064*water - 0.0014*water*water + 
               3.555 * salt_conc_water + 2.326 * salt_conc_water*salt_conc_water 
               -2.031 * salt_conc + 0.823 * salt_conc*salt_conc
               -0.071 * water * salt_conc_water + 0.033 * water * salt_conc 
               -2.754 * salt_conc_water * salt_conc)

p1 = (y1*price*10 )

fig, ax = plt.subplots()

ax.plot(water,p1, label = 'Profit', color = 'red') # profit is red line
ax.set_title('Profit and Yield by Water Depth')
ax.set_xlabel('Water Depth (cm)')
ax.set_ylabel('Profit ($/10 ha)')
ax.set_ylim([-100,7200])
ax.grid(True)
ax.legend(loc = 'lower right')

ax2 = ax.twinx()
ax2.plot(water,y1*lb_tonne, label = 'Yield') # yield is blue line
ax2.set_ylabel('Yield (tonnes/ha)')
ax.set_ylim([-100,8500])
ax2.legend()
plt.show()










In [ ]:
water = np.array(range(30))

y2 = (-3.350 + 0.2064*water - 0.0014*water*water + 
               3.555 * salt_conc_water + 2.326 * salt_conc_water*salt_conc_water 
               -2.031 * salt_conc + 0.823 * salt_conc*salt_conc
               -0.071 * water * salt_conc_water + 0.033 * water * salt_conc 
               -2.754 * salt_conc_water * salt_conc)

y2

In [ ]:
run = list(range(1000))
run = [x+1 for x in run]
run_c0 = list(range(4000))
run_c0 = [x+1 for x in run_c0]
years = 25
year = list(range(years))
year = [x+1 for x in year]
samples_c0 = len(run_c0)
samples = len(run)

len(run_c0)
len(year)


In [ ]:
# To help get tight bounds on variables later on

maxrev  = max(p1*10*ha)
minrev  = 0

#Water peaks profit and yield at 73

print('max irrigation water')
max_irrigation_water
print('cost cm / year capacity alt water')
cost_alt_water
print('cost irrigation water / cm')
cost_irrigation_water
print('irrigation water elc need per cm')
irrigation_water_elc
print('alt water elc need per cm')
alt_water_elc



Another useful bound to speed up computational time. The max price is as good as it gets and since the above code doesn't have any cost max price is max rev.

This code defines the charts that are used in the other Stochastic programs.

Defines the stochastic model. This model is simply a weighted average. Takes each possible precipatation and its corresponding yield and profit and averages them equally.

Variabels

alt_water; irrigation_water

water

crop_yield, profit

Objective functions are all profit and correspond to the different precipation scenarios. Need to change this to climaate scenarios.

Water is function of precipitation, alternative water and irrigation water.

Crop yield is function of saliniatity, soil, and water.


Showing all the results for each precipation scenario. Need to change this to climate scenario.

Showing results of Profit Expected Value for CVaR 95%.

The second model where the CVaR is 75%. Instead of a weighted average model which Takes each possible precipatation and its corresponding yield and profit and averages them equally. This model uses the CVaR method.


In [ ]:
t0a = time.perf_counter()


In [ ]:
m = gp.Model()

#m.params.NonConvex = 2 # Says its non convex but it is (FIXED quadratic constraints can't be equality)

# Set to maximize
m.ModelSense = -1

# Add variables

#First Stage Decision Variables Strategic Investment Decisions, pick what climate you are in which automatically picks how much alternative water to invest in

alt_water_cap = m.addVar(name = 'alt_water_cap', lb = 0) # cm of depth
alt_elc_cap = m.addVar(name = 'alt_elc_cap', lb = 0) # kW of depth


#Second Stage Decision Variables Operational Decisions, Pick how much water to use and how much irrigation water to buy

# Change these variables to be a function of option, climate, and season
water = m.addVars(year, lb = 0, name = 'water') # water used for crop, cm of depth
irrigation_water = m.addVars(year, lb = 0, ub = max_irrigation_water, name = 'irrigation_water') # cm of depth
alt_water = m.addVars(year, lb = 0, name = 'alt_water') # alt_water used for crop, cm of depth
alt_elc = m.addVars(year, lb = 0, name = 'alt_elc') # alt_elc to power water, kWh
utility_elc = m.addVars(year, lb = 0, name = 'util_elc') # util_elc to power water, kWh
elc = m.addVars(year, lb = 0, name = 'elc') # total elc to power water, kWh

#Functions of the Above, also functions of option, climate, and season, 
crop_yield = m.addVars(year, lb = 0, name = 'yield') # tonne/ha
profit = m.addVar(obj = 1, name = 'profit') # $



# Objective



m.addConstr( (profit == crop_yield.sum('*') * ha * price +
              -alt_water_cap * cost_alt_water  +
              -irrigation_water.sum('*') * cost_irrigation_water +
              -alt_elc_cap * cost_alt_elc +
              -utility_elc.sum('*') * cost_utility_elc
              )
            ,name = 'profit_constr')

# Water Equations

m.addConstrs( (alt_water[y] <= alt_water_cap
              for y in year)
             ,name = 'water_cap' )
 

m.addConstrs( ( alt_water[y] + irrigation_water[y] >= water[y] - rain_dict2[y]
              for y in year)
             ,name = 'water' )
             
             
#Energy for Alt Water Equations    

m.addConstrs( (alt_water[y] * alt_water_elc + irrigation_water[y] * irrigation_water_elc <= elc[y]
               for y in year)
             ,name = 'elc_water' )

# Energy Capacity and Balance Equations

m.addConstrs( (alt_elc[y] <= alt_elc_cap * cf_solar * hrs_month * gs_months
              for y in year)
             ,name = 'elc_cap' )




m.addConstrs( (alt_elc[y] + utility_elc[y] >= elc[y]
               for y in year)
             ,name = 'elc_balance' )





for y in year :
    m.addQConstr( (crop_yield[y] <= -3.350 + 0.2064*water[y]  - 0.0014*water[y]*water[y]  + 
       3.555 * salt_conc_water + 2.326 * salt_conc_water*salt_conc_water +
       -2.031 * salt_conc + 0.823 * salt_conc*salt_conc +
        -0.071 * water[y]  * salt_conc_water + 0.033 * water[y]  * salt_conc +
        -2.754 * salt_conc_water * salt_conc)
               , name = 'crop_yield' )


m.update();




In [ ]:
m.optimize()
obj = m.getObjective()
alt_water_ev = alt_water_cap.X
alt_elc_ev = alt_elc_cap.X
profit = profit.X

print('Objective Function:', obj.getValue())

print('Alt Water Investment:', alt_water_ev)

print('Alt ELC Investment:', alt_elc_ev)

print('Profit:', profit)


In [ ]:
# Writing Solutions to Dictionary

#profit_dict = m.getAttr('x', profit)
crop_yield_dict = m.getAttr('x', crop_yield)
water_dict = m.getAttr('x', water)
irrigation_water_dict = m.getAttr('x', irrigation_water)
alt_water_dict = m.getAttr('x', alt_water)
elc_dict = m.getAttr('x', elc)
utility_elc_dict = m.getAttr('x', utility_elc)
alt_elc_dict = m.getAttr('x', alt_elc)



In [ ]:
#Changing Tuple Dictionary to Dataframe


profit_df = pd.DataFrame({'run': [0], 'profit': profit})

crop_yield_df = pd.Series(crop_yield_dict).reset_index()
crop_yield_df.insert(loc = 0, column = 'run', value = 0)
crop_yield_df.columns = ['run', 'year', 'crop_yield'] 

rain_df = pd.Series(rain_c0_dict).reset_index()
rain_df.columns = ['run', 'year', 'water depth'] 

water_df = pd.Series(water_dict).reset_index()
water_df.insert(loc = 0, column = 'run', value = 0)
water_df.columns = ['run', 'year', 'water_depth'] 

irrigation_water_df = pd.Series(irrigation_water_dict).reset_index()
irrigation_water_df.insert(loc = 0, column = 'run', value = 0)
irrigation_water_df.columns = ['run', 'year', 'water_depth'] 

alt_water_df = pd.Series(alt_water_dict).reset_index()
alt_water_df.insert(loc = 0, column = 'run', value = 0)
alt_water_df.columns = ['run', 'year', 'water_depth'] 

alt_elc_df = pd.Series(alt_elc_dict).reset_index()
alt_elc_df.insert(loc = 0, column = 'run', value = 0)
alt_elc_df.columns = ['run', 'year', 'elc_prod'] 

elc_df = pd.Series(elc_dict).reset_index()
elc_df.insert(loc = 0, column = 'run', value = 0)
elc_df.columns = ['run', 'year', 'elc_prod'] 

utility_elc_df = pd.Series(utility_elc_dict).reset_index()
utility_elc_df.insert(loc = 0, column = 'run', value = 0)
utility_elc_df.columns = ['run', 'year', 'elc_prod'] 


In [ ]:
t1a = time.perf_counter()

elasped_ta = t1a - t0a

elasped_ta

In [ ]:
t0b = time.perf_counter()

In [ ]:
for s in run_c0:
    
    m = gp.Model()
    m.Params.OutputFlag = 0
    m.ModelSense = -1
    
    alt_water_cap = alt_water_ev # cm of depth
    alt_elc_cap = alt_elc_ev # kW of depth
    water = m.addVars(year, lb = 0, name = 'water') # water used for crop, cm of depth
    irrigation_water = m.addVars(year, lb = 0, ub = max_irrigation_water, name = 'irrigation_water') # cm of depth
    alt_water = m.addVars(year, lb = 0, name = 'alt_water') # alt_water used for crop, cm of depth
    alt_elc = m.addVars(year, lb = 0, name = 'alt_elc') # alt_elc to power water, kWh
    utility_elc = m.addVars(year, lb = 0, name = 'util_elc') # util_elc to power water, kWh
    elc = m.addVars(year, lb = 0, name = 'elc') # total elc to power water, kWh
    crop_yield = m.addVars(year, lb = 0, name = 'yield') # tonne/ha
    
    profit = m.addVar(obj = 1, name = 'profit') # $
    
    print('Start of Iteration:', s, 'Of 4000')
    
    m.addConstr( (profit == crop_yield.sum('*') * ha * price +
              -alt_water_cap * cost_alt_water  +
              -irrigation_water.sum('*') * cost_irrigation_water +
              -alt_elc_cap * cost_alt_elc +
              -utility_elc.sum('*') * cost_utility_elc
              )
            ,name = 'profit_constr');
    
    m.addConstrs( (alt_water[y] <= alt_water_cap
               for y in year)
             ,name = 'water_cap' );
 

    m.addConstrs( ( rain_c0_dict[s,y] + alt_water[y] + irrigation_water[y] >= water[y] 
              for y in year)
             ,name = 'water' );
    
    m.addConstrs( (alt_water[y] * alt_water_elc + irrigation_water[y] * irrigation_water_elc <= elc[y]
               for y in year)
             ,name = 'elc_water' );

# Energy Capacity and Balance Equations

    m.addConstrs( (alt_elc[y] <= alt_elc_cap * cf_solar * hrs_month * gs_months
              for y in year)
             ,name = 'elc_cap' );




    m.addConstrs( (alt_elc[y] + utility_elc[y] >= elc[y]
               for y in year)
             ,name = 'elc_balance' );





    for y in year :
        m.addQConstr( (crop_yield[y] <= -3.350 + 0.2064*water[y]  - 0.0014*water[y]*water[y]  + 
           3.555 * salt_conc_water + 2.326 * salt_conc_water*salt_conc_water +
           -2.031 * salt_conc + 0.823 * salt_conc*salt_conc +
            -0.071 * water[y]  * salt_conc_water + 0.033 * water[y]  * salt_conc +
            -2.754 * salt_conc_water * salt_conc)
               , name = 'crop_yield' );


    m.update();
   
    m.optimize()
    obj = m.getObjective()
    profit1 = profit.X
    print('Objective Function:', obj.getValue())
    print('Alt Water Investment:', alt_water_ev)
    print('Alt ELC Investment:', alt_elc_ev)
    print('Profit:' , profit1)
    print(s)
    
    crop_yield_dict = m.getAttr('x', crop_yield)
    water_dict = m.getAttr('x', water)
    irrigation_water_dict = m.getAttr('x', irrigation_water)
    alt_water_dict = m.getAttr('x', alt_water)
    elc_dict = m.getAttr('x', elc)
    utility_elc_dict = m.getAttr('x', utility_elc)
    alt_elc_dict = m.getAttr('x', alt_elc)
    
    profit_df = profit_df.append({'run': s, 'profit': profit1}, ignore_index = True)
    
    crop_yield_df1 = pd.Series(crop_yield_dict).reset_index()
    crop_yield_df1.insert(loc = 0, column = 'run', value = s)
    crop_yield_df1.columns = ['run', 'year', 'crop_yield'] 
    crop_yield_df = crop_yield_df.append(crop_yield_df1, ignore_index = True)
    
    water_df1 = pd.Series(water_dict).reset_index()
    water_df1.insert(loc = 0, column = 'run', value = s)
    water_df1.columns = ['run', 'year', 'water_depth'] 
    water_df = water_df.append(water_df1, ignore_index = True)
    

    irrigation_water_df1 = pd.Series(irrigation_water_dict).reset_index()
    irrigation_water_df1.insert(loc = 0, column = 'run', value = s)
    irrigation_water_df1.columns = ['run', 'year', 'water_depth'] 
    irrgiation_water_df = irrigation_water_df.append(irrigation_water_df1, ignore_index = True)

    alt_water_df1 = pd.Series(alt_water_dict).reset_index()
    alt_water_df1.insert(loc = 0, column = 'run', value = s)
    alt_water_df1.columns = ['run', 'year', 'water_depth'] 
    alt_water_df = alt_water_df.append(alt_water_df1, ignore_index = True)

    alt_elc_df1 = pd.Series(alt_elc_dict).reset_index()
    alt_elc_df1.insert(loc = 0, column = 'run', value = s)
    alt_elc_df1.columns = ['run', 'year', 'elc_prod'] 
    alt_elc_df = alt_elc_df.append(alt_elc_df1, ignore_index = True)
    

    elc_df1 = pd.Series(elc_dict).reset_index()
    elc_df1.insert(loc = 0, column = 'run', value = s)
    elc_df1.columns = ['run', 'year', 'elc_prod'] 
    elc_df = elc_df.append(elc_df1, ignore_index = True)

    utility_elc_df1 = pd.Series(utility_elc_dict).reset_index()
    utility_elc_df1.insert(loc = 0, column = 'run', value = s)
    utility_elc_df1.columns = ['run', 'year', 'elc_prod'] 
    utility_elc_df = utility_elc_df.append(utility_elc_df1, ignore_index = True)
    
    
    
    

    

In [ ]:
t1b = time.perf_counter()

elasped_tb = t1b - t0b

elasped_tb

In [1]:
profit_df
crop_yield_df
water_df
elc_df
alt_water_df
alt_elc_df
irrigation_water_df
utility_elc_df

NameError: name 'profit_df' is not defined

In [ ]:
# Calculating Confidence Interval

profit_array = np.array(list(profit_df['profit'].to_list()))




# rename profit array
a = profit_array * 1.0

# number of samples in array
n = len(a)
# calculate mean and sample error
m, se = np.mean(a), scs.sem(a)
# calculate t distrb
t = scs.t.ppf((1 + 0.95) / 2., n-1)
# calculate step size
h = se * t
# print sample size, mean, standard error, t value, step size, lower interval, and higher interval
n, m, se, t, h, m-h, m+h



In [ ]:
crop_yield_array = np.array(crop_yield_df['crop_yield'].to_list())

# rename crop yield array
b = crop_yield_array * 1.0

# number of samples in array
n = len(b)
# calculate mean and sample error
m, se = np.mean(b), scs.sem(b)
# calculate t distrb
t = scs.t.ppf((1 + 0.95) / 2., n-1)
# calculate step size
h = se * t
# print sample size, mean, standard error, t value, step size, lower interval, and higher interval
n, m, se, t, h, m-h, m+h


In [ ]:
profit_df.to_csv('profit_ev.csv', index=False) 
crop_yield_df.to_csv('crop_yield_ev.csv', index=False) 
water_df.to_csv('water_ev.csv', index=False) 
elc_df.to_csv('elc_ev.csv', index=False) 
irrigation_water_df.to_csv('irrigation_water_ev.csv', index=False) 
utility_elc_df.to_csv('utility_elc_ev.csv', index=False) 
alt_water_df.to_csv('alt_water_ev.csv', index=False) 
alt_elc_df.to_csv('alt_elc_ev.csv', index=False) 
